# slop-check — train on Kaggle (free T4) with Unsloth

1. Settings → Accelerator → **GPU T4 x1** (or P100), Internet **on**.
2. Add-ons → Secrets → add `HF_TOKEN` (a Hugging Face *write* token).
3. Edit `HF_REPO` below and run all cells (~1–2 h for 1.5B on ~10k examples).

In [ ]:
HF_REPO = "your-hf-username/slop-check-1.5b-gguf"   # <- change me
REPO = "https://github.com/thebunnyweb/slop-check"
MAX_STEPS = -1   # set e.g. 30 for a 5-minute smoke test

In [ ]:
!pip install -q unsloth
!git clone -q $REPO /kaggle/working/slop-check
%cd /kaggle/working/slop-check
!pip install -q -e ".[data]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# Free data: CPython stdlib (offline) + CommitPackFT (streamed). Injects known slop, so labels are exact.
!python data/build_dataset.py --source-dir "$(python -c 'import sysconfig;print(sysconfig.get_paths()["stdlib"])')" --commitpackft 30000 --max-examples 10000 --out data/out

In [ ]:
!python training/train_unsloth.py --data data/out --out runs/unsloth --max-steps $MAX_STEPS --gguf q4_k_m --push $HF_REPO

In [ ]:
# Sanity check the exported GGUF on the sample PR
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!slopcheck review --diff data/seed/sample_pr.diff --backend llamacpp --model $(ls runs/unsloth/gguf/*.gguf runs/unsloth/*.gguf 2>/dev/null | head -1)